# Analysis

Reads aggregate JSONs from `results/text_off/` and `results/text_on/`.


In [ ]:
import sys
sys.path.insert(0, "..")

import csv, os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import torch

from analysis.stats import load_results, summarise, paired_t

RESULTS_DIR = "../results"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

results = load_results(RESULTS_DIR)
print(f"loaded {len(results)} (model, dataset, use_text) entries")
for k in sorted(results): print(f"  {k}")


## Structural table (`use_text=False`)


In [ ]:
STRUCT_MODELS = ["gcn", "gat", "bigcn", "gps_mean", "gps_4way", "cascade_gps"]

print(f"{'Model':<14} {'Dataset':<12} {'Test F1 mean +/- std':<22} {'n':>3}")
print("-" * 55)
for model in STRUCT_MODELS:
    for dataset in ("twitter15", "twitter16"):
        entry = results.get((model, dataset, False))
        if entry is None:
            print(f"{model:<14} {dataset:<12} -")
        else:
            s = summarise(entry)
            print(f"{model:<14} {dataset:<12} {s['mean']:.3f} +/- {s['std']:.3f}{'':>10} {s['n']:>3}")


## Four structural contrasts (paired t-test)


In [ ]:
CONTRASTS = [
    ("GAT - GCN",             "gat",      "gcn"),
    ("GPS(mean) - GAT",        "gps_mean", "gat"),
    ("GPS(4way) - GPS(mean)",  "gps_4way", "gps_mean"),
    ("GPS(4way) - GAT",        "gps_4way", "gat"),
]

print(f"{'Contrast':<26} {'Dataset':<12} {'Delta F1':>8} {'std':>7} {'p':>10}")
print("-" * 68)
for name, ma, mb in CONTRASTS:
    for dataset in ("twitter15", "twitter16"):
        a = results.get((ma, dataset, False))
        b = results.get((mb, dataset, False))
        if not a or not b:
            print(f"{name:<26} {dataset:<12}   (missing)")
            continue
        t = paired_t(a, b)
        sig = " *" if t["p"] < 0.05 else ""
        print(f"{name:<26} {dataset:<12} {t['mean_delta']:+.3f}  {t['std_delta']:.3f}  {t['p']:.4f}{sig}")


## Text lift (text-on vs text-off, paired per seed)


In [ ]:
ALL_MODELS = ["gcn", "gat", "bigcn", "gps_4way", "cascade_gps"]

print(f"{'Model':<14} {'Dataset':<12} {'F1 off':>8} {'F1 on':>8} {'Delta F1':>9} {'p':>10}")
print("-" * 66)
for model in ALL_MODELS:
    for dataset in ("twitter15", "twitter16"):
        off = results.get((model, dataset, False))
        on  = results.get((model, dataset, True))
        if not off or not on:
            status = "(text-on missing)" if off else "(no results)"
            print(f"{model:<14} {dataset:<12} {status}")
            continue
        t = paired_t(on, off)
        sig = " *" if t["p"] < 0.05 else ""
        s_off = summarise(off)['mean']
        s_on  = summarise(on)['mean']
        print(f"{model:<14} {dataset:<12} {s_off:.3f}    {s_on:.3f}    {t['mean_delta']:+.3f}    {t['p']:.4f}{sig}")


## Cascade size buckets

Bucket counts for the test set. Per-bucket F1 requires per-cascade prediction dumps — extend `eval_epoch` to return predictions if needed.


In [ ]:
from data.dataset import CascadeDataset
from training.trainer import stratified_split

ds15 = CascadeDataset(root="../Twitter15_Twitter16", name="twitter15")
ds16 = CascadeDataset(root="../Twitter15_Twitter16", name="twitter16")

def bucket(n): return "<100" if n < 100 else ("100-500" if n <= 500 else ">500")

for ds, name in ((ds15, "twitter15"), (ds16, "twitter16")):
    _, _, test_idx = stratified_split(ds, split_seed=0)
    counts = {"<100": 0, "100-500": 0, ">500": 0}
    for i in test_idx: counts[bucket(ds[i].num_nodes)] += 1
    print(f"{name}: <100={counts['<100']}  100-500={counts['100-500']}  >500={counts['>500']}  total={len(test_idx)}")


## Attention visualization (GraphGPS, layer 1)

**Prereq**: checkpoint at `../results/text_off/gps_4way_twitter15_seed0.pt` — written by notebook 03 when `save_checkpoint=True`.


In [ ]:
import math
from collections import defaultdict

from torch_geometric.utils import to_dense_batch
from models.gps import GPSClassifier
from training.trainer import stratified_split
from data.dataset import LABEL_MAP

INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
DATASET = ds15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_, _, test_idx = stratified_split(DATASET, split_seed=0)
by_class = defaultdict(list)
for i in sorted(test_idx): by_class[int(DATASET[i].y.item())].append(i)
selected = [i for c in sorted(by_class) for i in by_class[c][:2]]
print("selected:", [(i, INV_LABEL[int(DATASET[i].y.item())], DATASET[i].num_nodes) for i in selected])

GPS_KWARGS = dict(in_channels=DATASET[0].x.shape[1], hidden_channels=128, num_layers=3,
                  heads=4, dropout=0.1, edge_dim=DATASET[0].edge_attr.shape[1], readout="4way")
model = GPSClassifier(**GPS_KWARGS).to(DEVICE)
ckpt = Path("../results/text_off/gps_4way_twitter15_seed0.pt")
if not ckpt.exists():
    raise FileNotFoundError(f"checkpoint not found: {ckpt}. Run notebook 03 with save_checkpoint=True.")
model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()

def layer1_attn_probs(data):
    data = data.to(DEVICE)
    with torch.no_grad():
        x = model.input_proj(data.x)
        batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
        dense, mask = to_dense_batch(x, batch)
        _, w = model.convs[0].attn(dense, dense, dense,
                                   key_padding_mask=~mask, need_weights=True, average_attn_weights=True)
    return w[0].cpu().numpy()

rows = []
for idx in selected:
    data = DATASET[idx]
    A = layer1_attn_probs(data)
    p_root = A[0]; p_root /= max(p_root.sum(), 1e-12)
    entropy = float(-np.sum(p_root * np.log(np.clip(p_root, 1e-12, None))))
    top3 = np.argsort(-p_root)[:3]
    rows.append({"idx": idx, "label": INV_LABEL[int(data.y.item())],
                 "N": int(data.num_nodes), "entropy": entropy,
                 "top3_depth_norm": data.x[:, 1].numpy()[top3].tolist()})
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(A, aspect="auto", cmap="viridis")
    ax.set_title(f"idx={idx} N={data.num_nodes} {INV_LABEL[int(data.y.item())]}")
    ax.set_xlabel("key node"); ax.set_ylabel("query node")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); plt.tight_layout()
    plt.savefig(FIG_DIR / f"attn_layer1_idx{idx}.png", dpi=120, bbox_inches="tight"); plt.close(fig)

for r in rows: print(r)


## Export tables to CSV


In [ ]:
with open("../results/table_structural.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "dataset", "test_f1_mean", "test_f1_std", "n"])
    for model in STRUCT_MODELS:
        for dataset in ("twitter15", "twitter16"):
            e = results.get((model, dataset, False))
            if not e: w.writerow([model, dataset, "", "", 0]); continue
            s = summarise(e)
            w.writerow([model, dataset, f"{s['mean']:.4f}", f"{s['std']:.4f}", s["n"]])
print("wrote table_structural.csv")

with open("../results/table_contrasts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["contrast", "dataset", "mean_delta", "std_delta", "t", "p", "n_pairs"])
    for name, ma, mb in CONTRASTS:
        for dataset in ("twitter15", "twitter16"):
            a = results.get((ma, dataset, False)); b = results.get((mb, dataset, False))
            if not a or not b: w.writerow([name, dataset] + [""] * 5); continue
            t = paired_t(a, b)
            w.writerow([name, dataset, f"{t['mean_delta']:.4f}", f"{t['std_delta']:.4f}", f"{t['t']:.4f}", f"{t['p']:.4f}", t["n_pairs"]])
print("wrote table_contrasts.csv")

with open("../results/table_text_lift.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "dataset", "f1_off", "f1_on", "mean_delta", "p", "n_pairs"])
    for model in ALL_MODELS:
        for dataset in ("twitter15", "twitter16"):
            off = results.get((model, dataset, False)); on = results.get((model, dataset, True))
            if not off or not on: w.writerow([model, dataset] + [""] * 5); continue
            t = paired_t(on, off)
            w.writerow([model, dataset, f"{summarise(off)['mean']:.4f}", f"{summarise(on)['mean']:.4f}", f"{t['mean_delta']:.4f}", f"{t['p']:.4f}", t["n_pairs"]])
print("wrote table_text_lift.csv")
